In [ ]:
import ctypes
import numpy as np
from pathlib import Path

lib = ctypes.CDLL("./mlp_lib.so")

lib.entrainer.argtypes = [
    ctypes.POINTER(ctypes.c_float),  # X
    ctypes.POINTER(ctypes.c_int),     # y
    ctypes.c_int,                     # n
    ctypes.c_int,                     # d
    ctypes.c_int,                     # H
    ctypes.c_int,                     # epochs
    ctypes.c_double,                  # lr
    ctypes.c_uint,                    # seed
    ctypes.POINTER(ctypes.c_double),  # W1_out
    ctypes.POINTER(ctypes.c_double),  # b1_out
    ctypes.POINTER(ctypes.c_double),  # W2_out
    ctypes.POINTER(ctypes.c_double),  # b2_out
]
lib.entrainer.restype = None

lib.predire_batch.argtypes = [
    ctypes.POINTER(ctypes.c_float),  # X_test
    ctypes.c_int,                     # n_test
    ctypes.c_int,                     # d
    ctypes.c_int,                     # H
    ctypes.POINTER(ctypes.c_double),  # W1
    ctypes.POINTER(ctypes.c_double),  # b1
    ctypes.POINTER(ctypes.c_double),  # W2
    ctypes.POINTER(ctypes.c_double),  # b2
    ctypes.POINTER(ctypes.c_int),     # predictions_out
]
lib.predire_batch.restype = None

In [ ]:
def lire_X(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        d = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        X = np.fromfile(f, dtype=np.float32, count=n * d)
    return np.ascontiguousarray(X.reshape(n, d)), n, d

def lire_y(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        y = np.fromfile(f, dtype=np.int32, count=n)
    return np.ascontiguousarray(y), n

racine = Path.cwd()
while racine != racine.parent and not (racine / "preprocessing").exists():
    racine = racine.parent
base = racine / "datasets" / "transformed" / "nb" / "normalisee"

X_train, n_train, d = lire_X(base / "X_train.f32bin")
y_train, _ = lire_y(base / "y_train.i32bin")
X_test, n_test, _ = lire_X(base / "X_test.f32bin")
y_test, _ = lire_y(base / "y_test.i32bin")
print(n_train, n_test, d)

In [ ]:
H = 32
epochs = 30
lr = 0.001
seed = 67
K_CLASSES = 3

W1 = np.zeros(H * d, dtype=np.float64)
b1 = np.zeros(H, dtype=np.float64)
W2 = np.zeros(K_CLASSES * H, dtype=np.float64)
b2 = np.zeros(K_CLASSES, dtype=np.float64)

pf = ctypes.POINTER(ctypes.c_float)
pd = ctypes.POINTER(ctypes.c_double)
pi = ctypes.POINTER(ctypes.c_int)

lib.entrainer(
    X_train.ctypes.data_as(pf), y_train.ctypes.data_as(pi),
    n_train, d, H, epochs, lr, seed,
    W1.ctypes.data_as(pd), b1.ctypes.data_as(pd),
    W2.ctypes.data_as(pd), b2.ctypes.data_as(pd),
)

predictions = np.zeros(n_test, dtype=np.int32)
lib.predire_batch(
    X_test.ctypes.data_as(pf), n_test, d, H,
    W1.ctypes.data_as(pd), b1.ctypes.data_as(pd),
    W2.ctypes.data_as(pd), b2.ctypes.data_as(pd),
    predictions.ctypes.data_as(pi),
)

acc_test = (predictions == y_test).mean()
print("acc test (lib ctypes) :", round(acc_test, 3))